In [ ]:
import sys
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.gridspec import GridSpec
sys.path.insert(0, '.')
from utils import fill_deaths, SMALL_CONSTANT, DATA_DIR, OUT_DIR

# ── Paths ──────────────────────────────────────────────────────────────────
OUT_DIR.mkdir(exist_ok=True)

CSV_PATH      = DATA_DIR / 'violence_aid_merged.csv'
ADM2_PATH     = DATA_DIR / 'cod_admin_boundaries.shp/cod_admin2.shp'
ADM1_PATH     = DATA_DIR / 'cod_admin_boundaries.shp/cod_admin1.shp'
OUT_PNG_START = OUT_DIR  / 'drc_choropleth_start.png'
OUT_PNG_END   = OUT_DIR  / 'drc_choropleth_end.png'
OUT_GIF       = OUT_DIR  / 'drc_choropleth.gif'

BG = 'white'


In [2]:
# ── Load & prep ────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
df['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m')
df = df.sort_values(['admin2_name', 'year_month'])

adm2 = gpd.read_file(ADM2_PATH).to_crs('EPSG:4326')
adm1 = gpd.read_file(ADM1_PATH).to_crs('EPSG:4326')

print(f'Rows: {len(df):,} | Admin-2: {len(adm2)} | Admin-1: {len(adm1)}')

Rows: 10,512 | Admin-2: 164 | Admin-1: 26


In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────
# fill_deaths imported from utils.py

def style_colorbar(cbar, label):
    """Apply consistent black-on-white styling to a colorbar."""
    cbar.ax.yaxis.set_tick_params(color='black')
    cbar.outline.set_edgecolor('black')
    plt.setp(cbar.ax.get_yticklabels(), color='black', fontsize=8)
    cbar.set_label(label, color='black', fontsize=9, labelpad=8)


def make_static(q, adm2, adm1, qdf, cmap, norm, path):
    """Render a single-quarter choropleth PNG."""
    sub   = qdf[qdf['quarter'] == q][['admin2_name', 'metric']]
    frame = adm2.merge(sub, left_on='adm2_name', right_on='admin2_name', how='left')

    fig, ax = plt.subplots(figsize=(15, 13))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)

    frame.plot(column='metric', ax=ax, cmap=cmap, norm=norm,
               linewidth=0.15, edgecolor='#cccccc',
               missing_kwds={'color': '#dddddd', 'label': 'No data'})
    adm1.boundary.plot(ax=ax, color='#333333', linewidth=1.6, alpha=0.9, zorder=3)

    ax.set_title(
        f'DRC Aid Efficiency — {q}\nLog(Aid Spend) − Log(Deaths) by Territory',
        color='black', fontsize=14, fontweight='bold', pad=12,
    )
    ax.set_axis_off()

    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation='vertical',
                        fraction=0.022, pad=0.02, shrink=0.62)
    style_colorbar(cbar, 'log($/death)   higher = more aid per death')

    ax.text(0.01, 0.01,
            'Light yellow = high $/death (over-served)\nDark crimson = low $/death (under-served)\nLight gray = no data',
            transform=ax.transAxes, color='black', fontsize=8, va='bottom',
            bbox=dict(boxstyle='round,pad=0.35', facecolor='white', edgecolor='#cccccc', alpha=0.9))

    fig.savefig(str(path), dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    print(f'Saved {path}')


# ── Apply death-fill imputation ────────────────────────────────────────────
df['deaths_filled'] = df.groupby('admin2_name')['total_deaths'].transform(fill_deaths)
print('Death fill done.')


In [4]:
# ── Quarterly aggregation & log $/death ────────────────────────────────────
df['quarter'] = df['year_month'].dt.to_period('Q')

qdf = (
    df.groupby(['admin2_name', 'quarter'])
    .agg(aid_spend=('total_aid_spend', 'sum'),
         deaths_filled=('deaths_filled', 'sum'))
    .reset_index()
)

# log(aid/deaths) in log-space = log1p(aid) - log(deaths)
qdf['log_aid']    = np.log1p(qdf['aid_spend'])
qdf['log_deaths'] = np.log(qdf['deaths_filled'].clip(lower=SMALL_CONSTANT))
qdf['metric']     = qdf['log_aid'] - qdf['log_deaths']

quarters = sorted(qdf['quarter'].unique())
print(f'{len(quarters)} quarters | metric range [{qdf["metric"].min():.2f}, {qdf["metric"].max():.2f}]')
print(qdf['metric'].describe())

24 quarters | metric range [-5.32, 19.15]
count    3504.000000
mean       11.190144
std         6.619181
min        -5.318120
25%        10.397357
50%        13.799271
75%        16.229827
max        19.154588
Name: metric, dtype: float64


In [5]:
# ── Save quarterly metric for downstream notebooks (08) ───────────────────
metric_path = DATA_DIR / 'drc_quarterly_metric.csv'
qdf.to_csv(metric_path, index=False)
print(f'Saved quarterly metric → {metric_path}')


Saved quarterly metric → /Users/jackzipper/QSS20/final_project/final_project_data/drc_quarterly_metric.csv


In [6]:
# ── Colour scale ───────────────────────────────────────────────────────────
# The metric is bimodal: a large cluster near 0 (no aid, deaths present)
# and a spread from ~10-19 (aid present). Use quantile boundaries so every
# color band covers an equal share of the distribution, making territories
# visually distinguishable rather than uniformly dark.

N_BINS = 9
quantile_bounds = np.quantile(qdf['metric'].dropna(),
                              np.linspace(0, 1, N_BINS + 1))
# Deduplicate boundaries (can happen at flat regions)
quantile_bounds = np.unique(quantile_bounds)

# Light yellow (high $/death, over-served) → dark crimson (low $/death, under-served)
colors_hi_to_lo = [
    '#ffffcc', '#fee5d9', '#fcae91', '#fc7050',
    '#ef3b2c', '#cb181d', '#a50f15', '#800026', '#67000d',
]
# Trim to match number of bins
colors_hi_to_lo = colors_hi_to_lo[:len(quantile_bounds) - 1]
# Reverse: low values → dark red end
colors_lo_to_hi = list(reversed(colors_hi_to_lo))

cmap = mcolors.ListedColormap(colors_lo_to_hi)
norm = mcolors.BoundaryNorm(quantile_bounds, ncolors=len(colors_lo_to_hi))

print(f'Quantile boundaries: {np.round(quantile_bounds, 2)}')

Quantile boundaries: [-5.32 -0.41  6.53 11.94 13.13 14.29 15.49 16.28 16.92 19.15]


In [7]:
# ── Static PNGs: first and last quarter of the analysis ───────────────────
make_static(quarters[0],  adm2, adm1, qdf, cmap, norm, OUT_PNG_START)
make_static(quarters[-1], adm2, adm1, qdf, cmap, norm, OUT_PNG_END)


Saved /Users/jackzipper/QSS20/final_project/output/drc_choropleth_start.png


Saved /Users/jackzipper/QSS20/final_project/output/drc_choropleth_end.png


In [ ]:
# ── Animated GIF — fixed-size axes, colorbar on a dedicated subplot ────────
from matplotlib.gridspec import GridSpec

fig_a = plt.figure(figsize=(15, 12))
fig_a.patch.set_facecolor('white')
gs = GridSpec(1, 2, figure=fig_a, width_ratios=[22, 1], wspace=0.03)
ax_map  = fig_a.add_subplot(gs[0, 0])
ax_cbar = fig_a.add_subplot(gs[0, 1])

# Draw the colorbar once on its dedicated axes — it never moves
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig_a.colorbar(sm, cax=ax_cbar)
style_colorbar(cbar, 'log($/death)')
ax_cbar.yaxis.set_tick_params(labelcolor='black')
ax_cbar.set_facecolor('white')

def animate(i):
    ax_map.clear()
    ax_map.set_facecolor('white')

    q = quarters[i]
    sub = qdf[qdf['quarter'] == q][['admin2_name', 'metric']]
    frame = adm2.merge(sub, left_on='adm2_name', right_on='admin2_name', how='left')

    frame.plot(
        column='metric', ax=ax_map,
        cmap=cmap, norm=norm,
        linewidth=0.15, edgecolor='#cccccc',
        missing_kwds={'color': '#dddddd'},
    )
    adm1.boundary.plot(ax=ax_map, color='#333333', linewidth=1.6, alpha=0.9, zorder=3)

    ax_map.set_title(
        f'DRC Aid Efficiency — {q}\nLog(Aid Spend) − Log(Deaths) by Territory',
        color='black', fontsize=12, fontweight='bold', pad=10,
    )
    ax_map.set_axis_off()
    ax_map.text(0.01, 0.01,
                'Light yellow = high $/death\nDark crimson = low $/death',
                transform=ax_map.transAxes, color='black', fontsize=8, va='bottom',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#cccccc', alpha=0.9))

print(f'Building {len(quarters)}-frame animation...')
ani = FuncAnimation(fig_a, animate, frames=len(quarters), interval=600, repeat=True)
ani.save(str(OUT_GIF), writer=PillowWriter(fps=2), dpi=110)
plt.close(fig_a)
print(f'GIF saved → {OUT_GIF}')


In [9]:
import os
print(f'Start PNG : {os.path.getsize(OUT_PNG_START)//1024} KB  ({quarters[0]})')
print(f'End PNG   : {os.path.getsize(OUT_PNG_END)//1024} KB  ({quarters[-1]})')
print(f'GIF       : {os.path.getsize(OUT_GIF)//1024} KB')


Start PNG : 479 KB  (2021Q1)
End PNG   : 479 KB  (2026Q4)
GIF       : 2356 KB
